# Experiment 025: the 7B exposure-sweep ends (Colab)

Six base MC-SFT arms completing the 7B supervised exposure curve: p2 (31
steps) and p32 (490 steps) at seeds 0, 1 and 2, exact 014 mirrors except the
step count. Registered questions in the 025 plan: does the flat exposure
curve replicate at 7B, and does heavy exposure erode the probe the way it
eroded the 0.5B hard probe. The full arm set is locked and bug-gated only.

## 0. Compute units and mount Drive

Fill `cu_before` and `unit_rate` now.

In [ ]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = None   # fill from the resources panel
unit_rate = None   # fill from the resources panel
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close an unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '025_colab_b7_exposure_ends',
                 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260802_001712 | cu_before = 59.46
Mounted at /content/drive
saved open session: 20260802_001712
drive mounted -> /content/drive/MyDrive/inm363


## 1. GPU check: A100 with 80GB required

In [ ]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime for the 7b supervised runs'


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-e1b1677d-00f0-d47b-b724-6941f4b4d9b4)
total vram: 79.3 GiB


## 2. Clone or update the repo

In [13]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 4.23 KiB | 1.41 MiB/s, done.
From https://github.com/oleh-hastov/INM363-IndividualProject-Dissertation
   7b0a29c..8f0f87e  main       -> origin/main
Updating 7b0a29c..8f0f87e
Fast-forward
 001_Code/notebooks/025_colab_b7_exposure_ends.ipynb | 2 +-
 001_Code/tests/test_024_supervised_extension.py     | 3 +++
 2 files changed, 4 insertions(+), 1 deletion(-)
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [ ]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# colab includes a torchao version that is incompatible with this peft version
# no quantisation is used in these runs
!pip uninstall -q -y torchao
print('dependencies installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 141.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.6 MB/s eta 0:00:00
dependencies installed


## 4. W&B key

Grant this notebook access to `WANDB_API_KEY` in Colab secrets.
The prompt is used only when the W&B secret is unavailable. This key is only used for experiment tracking.


In [ ]:
import getpass, os
wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None
if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')
os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check Drive space

In [ ]:
import os, shutil
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 40, 'need at least 40GB free on drive for the six supervised arms'


open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260802_001712.json
drive free: 179 GB


## 6. Build the data splits

In [ ]:
!python scripts/build_data.py

README.md: 100% 7.93k/7.93k [00:00<00:00, 20.2MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes:  64% 1.47M/2.31M [00:00<00:00, 2.28MB/s, 28.7kB/s  ]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:00<00:00, 2.75MB/s,  226kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:00<00:00, 2.76MB/s,  227kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:00<00:00, 704kB/s, 41.5kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:00<00:00, 704kB/s, 41.5kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 387753.25 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 297434.78 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 78.1MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00

## 7. Select the staged arms

Both exposures always run together; only the seed list stages.

In [ ]:
# SEEDS_TO_RUN = (0,)
SEEDS_TO_RUN = (0, 1, 2)
EXPOSURES = ('p2', 'p32')
STEPS = {'p2': 31, 'p32': 490}

def config_path(exp, seed):
    return f'configs/025_{exp}_colab_b7_seed{seed}.yaml'

def run_tag(exp, seed):
    return f'025_{exp}_COLAB_FULL_B7_sft_only_seed{seed}'

JOBS = [(exp, seed, config_path(exp, seed), run_tag(exp, seed))
        for seed in SEEDS_TO_RUN for exp in EXPOSURES]
print('arms this pass:', [(exp, seed) for exp, seed, _, _ in JOBS])

arms this pass: [('p2', 0), ('p32', 0), ('p2', 1), ('p32', 1), ('p2', 2), ('p32', 2)]


## 8. Preflight: bind the code, reuse the 014 base probe, verify sources

The base probe copies keep the runner from touching the test set again; the
execution binding pins both staged passes to the same scientific code.

In [14]:
import glob, hashlib, json, os, shutil, sys
sys.path.insert(0, os.getcwd())
from checkpoint import _config_hash
from config import load_config
from obs import _git_info, _versions
from train.model_utils import resolve_dtype

D = '/content/drive/MyDrive/inm363'

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

# bind both staged passes to the same notebook and scientific code
with open('notebooks/025_colab_b7_exposure_ends.ipynb') as f:
    notebook = json.load(f)
protocol_source = ''.join(
    line for cell in notebook['cells'] for line in cell.get('source', [])
    if 'SEEDS_TO_RUN =' not in line)
scientific_paths = ['config.py', 'checkpoint.py', 'enums.py', 'obs.py',
                    'scripts/build_data.py', 'scripts/env_switch.py',
                    'scripts/run.py', 'scripts/eval_adapter.py']
for folder in ('analysis', 'data', 'eval', 'train'):
    scientific_paths.extend(
        os.path.relpath(path) for path in glob.glob(f'{folder}/**/*.py', recursive=True))
scientific_paths = sorted(set(scientific_paths))
all_configs = sorted(glob.glob('configs/025_*_colab_b7_seed*.yaml'))
execution_protocol = {
    'notebook_protocol_sha256': hashlib.sha256(protocol_source.encode()).hexdigest(),
    'scientific_file_hashes': {path: sha(path) for path in scientific_paths},
    'config_hashes': {path: _config_hash(load_config(path)) for path in all_configs},
    # the core hash skips eval, log and exec on purpose, so the raw files are
    # hashed too and nothing in the yamls can drift between passes
    'config_file_hashes': {path: sha(path) for path in all_configs}}
execution_sha = hashlib.sha256(
    json.dumps(execution_protocol, sort_keys=True).encode()).hexdigest()
execution_path = os.path.join(D, 'results', '025_execution_binding.json')
if os.path.exists(execution_path):
    saved = json.load(open(execution_path))
    assert saved['protocol'] == execution_protocol, '025 scientific code changed between staged passes'
else:
    tmp = execution_path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump({'created_git': _git_info(), 'protocol_sha256': execution_sha,
                   'protocol': execution_protocol}, f, indent=2)
    os.replace(tmp, execution_path)
print('025 staged execution binding verified')

# reuse the existing 014 raw-base probe instead of touching the test set again
base_source = os.path.join(D, 'results', 'base_evals', '014_p8_COLAB_FULL_B7')
probe_files = ('base_logprob.json', 'base_logprob_preds.jsonl')
source_hashes = {}
for fname in probe_files:
    p = os.path.join(base_source, fname)
    assert os.path.exists(p), f'014 base probe file missing: {p}'
    source_hashes[fname] = sha(p)
assert json.load(open(os.path.join(base_source, 'base_logprob.json')))['n_items'] == 1884
with open(os.path.join(base_source, 'base_logprob_preds.jsonl')) as f:
    assert sum(1 for line in f if line.strip()) == 1884
src_man = json.load(open(os.path.join(D, 'results', '014_p8_COLAB_FULL_B7_sft_only_seed0', 'manifest.json')))
assert src_man['model_id'] == 'Qwen/Qwen2.5-7B' and src_man['resolved_dtype'] == 'bfloat16'
for dataset in ('cais/mmlu@', 'openai/gsm8k@'):
    assert src_man['versions'].get(dataset) == _versions().get(dataset)
reuse_record = {'source': 'results/base_evals/014_p8_COLAB_FULL_B7',
                'files': source_hashes, 'n_items': 1884,
                'source_manifest_run': '014_p8_COLAB_FULL_B7_sft_only_seed0',
                'source_git_commit': src_man['git']['commit'],
                'scope': 'raw 7b base probe, generative base eval not reused'}
for exp in EXPOSURES:
    dest = os.path.join(D, 'results', 'base_evals', f'025_{exp}_COLAB_FULL_B7')
    os.makedirs(dest, exist_ok=True)
    for fname in probe_files:
        a, b = os.path.join(base_source, fname), os.path.join(dest, fname)
        if os.path.exists(b):
            assert sha(b) == source_hashes[fname], f'{exp}: existing base probe differs from 014'
        else:
            tmp = b + '.tmp'
            shutil.copy2(a, tmp)
            assert sha(tmp) == source_hashes[fname]
            os.replace(tmp, b)
    rp = os.path.join(dest, 'base_probe_reuse.json')
    if os.path.exists(rp):
        assert json.load(open(rp)) == reuse_record, f'{exp}: reuse record changed'
    else:
        tmp = rp + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(reuse_record, f, indent=2)
        os.replace(tmp, rp)
    print(f'{exp}: 014 base probe reused and hash-verified')

# per-arm config and environment checks against the recorded 014 parent
for exp, seed, cfg_path, tag in JOBS:
    cfg = load_config(cfg_path)
    assert cfg.train.seed == seed and cfg.train.max_steps == STEPS[exp]
    assert cfg.train.sft_batch_size * cfg.train.sft_grad_accum == 16
    assert not cfg.train.sft_from_adapter
    assert cfg.exec.allow_oom_fallback is False
    assert cfg.log.wandb_project == 'inm363-grpo'
    cur = _versions()
    for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate', 'cais/mmlu@', 'openai/gsm8k@'):
        assert cur.get(lib) == src_man['versions'].get(lib), f'{tag}: {lib} differs from 014'
    assert resolve_dtype(cfg) == 'bfloat16'
    out = os.path.join(D, 'results', tag)
    os.makedirs(out, exist_ok=True)
    source = {'exposure': exp, 'seed': seed, 'config': cfg_path,
              'config_hash': _config_hash(cfg), 'config_file_sha256': sha(cfg_path),
              'execution_protocol_sha256': execution_sha,
              'parent_reference': '014_p8_COLAB_FULL_B7'}
    sp = os.path.join(out, 'source_binding.json')
    if os.path.exists(sp):
        assert json.load(open(sp)) == source, f'{tag}: source binding changed'
    else:
        with open(sp, 'w') as f:
            json.dump(source, f, indent=2)
    print(f'{tag}: verified')

025 staged execution binding verified
p2: 014 base probe reused and hash-verified
p32: 014 base probe reused and hash-verified
025_p2_COLAB_FULL_B7_sft_only_seed0: verified
025_p32_COLAB_FULL_B7_sft_only_seed0: verified
025_p2_COLAB_FULL_B7_sft_only_seed1: verified
025_p32_COLAB_FULL_B7_sft_only_seed1: verified
025_p2_COLAB_FULL_B7_sft_only_seed2: verified
025_p32_COLAB_FULL_B7_sft_only_seed2: verified


## 9. Wiring check

In [15]:
import subprocess, sys
for _, _, cfg_path, tag in JOBS:
    print('wiring:', tag)
    subprocess.run([sys.executable, 'scripts/run.py', '--config', cfg_path], check=True)

wiring: 025_p2_COLAB_FULL_B7_sft_only_seed0
wiring: 025_p32_COLAB_FULL_B7_sft_only_seed0
wiring: 025_p2_COLAB_FULL_B7_sft_only_seed1
wiring: 025_p32_COLAB_FULL_B7_sft_only_seed1
wiring: 025_p2_COLAB_FULL_B7_sft_only_seed2
wiring: 025_p32_COLAB_FULL_B7_sft_only_seed2


## 10. Train the selected arms and run the probes

Finished arms skip before the runner so their manifests and the ledger stay
exactly as their own sessions wrote them.

In [16]:
import os
from checkpoint import _config_hash
from config import load_config

def training_probe_complete(cfg_path, tag):
    cfg = load_config(cfg_path)
    out = os.path.join(D, 'results', tag)
    adapter = os.path.join(D, 'checkpoints', tag, 'sft')
    needed = [os.path.join(adapter, name) for name in
              ('adapter_model.safetensors', 'adapter_config.json', 'adapter_base.json',
               'config_hash.txt')]
    needed.extend(os.path.join(out, name) for name in
                  ('sft_trainlog.json', 'sft_logprob.json', 'sft_logprob_preds.jsonl',
                   'manifest.json', 'summary.json'))
    if not all(os.path.exists(path) for path in needed):
        return False
    return open(os.path.join(adapter, 'config_hash.txt')).read().strip() == _config_hash(cfg)

assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in section 0'
for exp, seed, cfg_path, tag in JOBS:
    if training_probe_complete(cfg_path, tag):
        print(f'{tag}: training and probe complete, skipped')
        continue
    print(f'===== train {tag} =====')
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config {cfg_path} --stage all
    if _exit_code != 0:
        raise SystemExit(f'{tag} failed with exit code {_exit_code}; keep the declared settings unchanged')


025_p2_COLAB_FULL_B7_sft_only_seed0: training and probe complete, skipped
025_p32_COLAB_FULL_B7_sft_only_seed0: training and probe complete, skipped
===== train 025_p2_COLAB_FULL_B7_sft_only_seed1 =====
2026-08-02 00:39:32,866 INFO using wandb key from env
2026-08-02 00:39:32,867 INFO run 025_p2_COLAB_FULL_B7_sft_only_seed1 device=cuda stages=['base_eval', 'sft', 'sft_eval'] trackers=['tensorboard', 'wandb'] out=/content/drive/MyDrive/inm363/results/025_p2_COLAB_FULL_B7_sft_only_seed1
2026-08-02 00:39:32,867 INFO base_eval skipped, probe-only run
2026-08-02 00:39:32,868 INFO base_logprob_eval already done, skipping
config.json: 100% 686/686 [00:00<00:00, 3.96MB/s]
tokenizer_config.json: 100% 7.23k/7.23k [00:00<00:00, 23.8MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 94.9MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 134MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 138MB/s]
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 78.7MB/s]
Reconstructing (incomplete tota

## 11. Training and provenance gates

In [17]:
import glob, json, os
from collections import Counter
from checkpoint import _config_hash
from config import load_config

D = '/content/drive/MyDrive/inm363'
for exp, seed, cfg_path, tag in JOBS:
    cfg = load_config(cfg_path)
    n = STEPS[exp]
    result_dir = os.path.join(D, 'results', tag)
    adapter_dir = os.path.join(D, 'checkpoints', tag, 'sft')
    for fname in ('adapter_model.safetensors', 'adapter_config.json', 'adapter_base.json'):
        assert os.path.exists(os.path.join(adapter_dir, fname)), f'{tag}: missing {fname}'
    hp = os.path.join(adapter_dir, 'config_hash.txt')
    assert os.path.exists(hp) and open(hp).read().strip() == _config_hash(cfg)

    history = json.load(open(os.path.join(result_dir, 'sft_trainlog.json')))
    steps = [(int(r['step']), r) for r in history if 'step' in r and 'learning_rate' in r and 'loss' in r]
    counts = Counter(s for s, _ in steps)
    assert set(counts) == set(range(1, n + 1)) and all(v == 1 for v in counts.values()), \
        f'{tag}: step coverage is not 1..{n}'
    lrs = [r['learning_rate'] for _, r in sorted(steps)]
    # the last logged lr under this scheduler is peak/steps, gate with headroom
    assert 9.9e-5 <= lrs[0] <= 1.0e-4 and lrs[-1] <= 1.1e-4 / n, f'{tag}: schedule ends wrong'
    assert all(b <= a + 1e-12 for a, b in zip(lrs, lrs[1:])), f'{tag}: learning rate increased'

    prov = json.load(open(os.path.join(adapter_dir, 'adapter_base.json')))
    assert prov == {'kind': 'hf_id', 'value': 'Qwen/Qwen2.5-7B'}, f'{tag}: wrong provenance'
    probe = json.load(open(os.path.join(result_dir, 'sft_logprob.json')))
    assert probe['n_items'] == 1884
    with open(os.path.join(result_dir, 'sft_logprob_preds.jsonl')) as f:
        assert sum(1 for line in f if line.strip()) == 1884
    manifest = json.load(open(os.path.join(result_dir, 'manifest.json')))
    assert manifest['seed'] == seed and manifest['config']['train']['max_steps'] == n
    print(f'{tag}: {n} steps, schedule, probe and provenance passed')

025_p2_COLAB_FULL_B7_sft_only_seed0: 31 steps, schedule, probe and provenance passed
025_p32_COLAB_FULL_B7_sft_only_seed0: 490 steps, schedule, probe and provenance passed
025_p2_COLAB_FULL_B7_sft_only_seed1: 31 steps, schedule, probe and provenance passed
025_p32_COLAB_FULL_B7_sft_only_seed1: 490 steps, schedule, probe and provenance passed
025_p2_COLAB_FULL_B7_sft_only_seed2: 31 steps, schedule, probe and provenance passed
025_p32_COLAB_FULL_B7_sft_only_seed2: 490 steps, schedule, probe and provenance passed


## 12. Generative evaluations

In [18]:
import hashlib, json, os

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

D = '/content/drive/MyDrive/inm363'
for exp, seed, cfg_path, tag in JOBS:
    out = os.path.join(D, 'results', tag)
    adapter = os.path.join(D, 'checkpoints', tag, 'sft')
    binding = {name: sha(os.path.join(adapter, name)) for name in
               ('adapter_model.safetensors', 'adapter_config.json', 'adapter_base.json')}
    bind_path = os.path.join(out, 'sft_eval_binding.json')
    generative_exists = any(os.path.exists(os.path.join(out, name)) for name in
                            ('sft_eval.json', 'sft_preds.jsonl'))
    if os.path.exists(bind_path):
        assert json.load(open(bind_path)) == binding, f'{tag}: evaluation binding changed'
    else:
        assert not generative_exists, f'{tag}: generative outputs exist without a binding'
        with open(bind_path, 'w') as f:
            json.dump(binding, f, indent=2)

    print(f'===== evaluate {tag} =====')
    !python scripts/eval_adapter.py --config {cfg_path} --adapter {adapter} --condition sft --out {out} --seed {seed}
    if _exit_code != 0:
        raise SystemExit(f'{tag} evaluation failed with exit code {_exit_code}')

    ev = json.load(open(os.path.join(out, 'sft_eval.json')))
    lp = json.load(open(os.path.join(out, 'sft_logprob.json')))
    assert ev['n_items'] == 1884 and lp['n_items'] == 1884
    for pred_file in ('sft_preds.jsonl', 'sft_logprob_preds.jsonl'):
        with open(os.path.join(out, pred_file)) as f:
            rows = sum(1 for line in f if line.strip())
        assert rows == 1884, f'{tag}: {pred_file} has {rows} rows'
    eval_manifest = json.load(open(os.path.join(out, 'sft_eval_manifest.json')))
    assert eval_manifest.get('completed') is True
    print(f'{tag}: evaluation complete and bound')


===== evaluate 025_p2_COLAB_FULL_B7_sft_only_seed0 =====
sft gen eval already done, skipping
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.8828213879408419, 'medium': 0.7635467980295566, 'hard': 0.547979797979798}
probe per-tier: {'easy': 0.8976109215017065, 'medium': 0.80623973727422, 'hard': 0.5707070707070707}
025_p2_COLAB_FULL_B7_sft_only_seed0: evaluation complete and bound
===== evaluate 025_p32_COLAB_FULL_B7_sft_only_seed0 =====
sft gen eval already done, skipping
sft probe already done, skipping
summary completed with the new evals
gen per-tier: {'easy': 0.888509670079636, 'medium': 0.7881773399014779, 'hard': 0.553030303030303}
probe per-tier: {'easy': 0.8953356086461889, 'medium': 0.8095238095238095, 'hard': 0.5656565656565656}
025_p32_COLAB_FULL_B7_sft_only_seed0: evaluation complete and bound
===== evaluate 025_p2_COLAB_FULL_B7_sft_only_seed1 =====
Loading weights: 100% 339/339 [00:00<00:00, 5570.08it/s]
sft eval (1884 items)

## 13. Registered comparison

Binding only when all six arms are complete; families are the two exposures,
Holm across seeds inside each, decisive-only common readings, probe against
the recorded raw-base probe, and the descriptive four-point exposure curve. The curve is a
method-as-practised comparison: each exposure carries its own full
learning-rate decay, so points differ in schedule as well as passes.

In [19]:
import json, os, sys
sys.path.insert(0, os.getcwd())
from analysis.stats import _read_preds, align_pred_rows, holm, mcnemar_exact, paired_delta_ci
from data.subjects import TIERS, tier_of

D = '/content/drive/MyDrive/inm363'
NEEDED = ('sft_eval.json', 'sft_preds.jsonl', 'sft_logprob.json', 'sft_logprob_preds.jsonl')

def preds(tag, cond):
    return _read_preds(os.path.join(D, 'results', tag, f'{cond}_preds.jsonl'))

def acc(rows):
    return sum(r['correct'] for r in rows) / len(rows)

def hard_acc(rows):
    xs = [r['correct'] for r in rows if tier_of(r['subject']) == 'hard']
    return sum(xs) / len(xs)

def metric(new_rows, ref_rows):
    na, ra = align_pred_rows(new_rows, ref_rows)
    nc, rc = [r['correct'] for r in na], [r['correct'] for r in ra]
    d, lo, hi = paired_delta_ci(rc, nc)
    n01 = sum(not a and b for a, b in zip(rc, nc))
    n10 = sum(a and not b for a, b in zip(rc, nc))
    return {'reference_acc': acc(ra), 'new_acc': acc(na), 'delta': d, 'ci': [lo, hi],
            'p_raw': mcnemar_exact(n01, n10)}

def classify(row, p_adj):
    if abs(row['delta']) >= 0.02 and p_adj < 0.05:
        return 'material improvement' if row['delta'] > 0 else 'material decline'
    if row['ci'][0] > -0.02 and row['ci'][1] < 0.02:
        return 'equivalent'
    return 'inconclusive'

def common_reading(cats):
    decisive = ('equivalent', 'material improvement', 'material decline')
    if len(set(cats)) == 1 and cats[0] in decisive:
        return cats[0]
    return 'mixed or inconclusive, report arm by arm'

done_all = all(os.path.exists(os.path.join(D, 'results', f'025_{e}_COLAB_FULL_B7_sft_only_seed{s}', f))
               for e in EXPOSURES for s in (0, 1, 2) for f in NEEDED)
if not done_all:
    print('PROVISIONAL: registered categories wait for all six arms')

base_probe = _read_preds(os.path.join(D, 'results', 'base_evals', '025_p2_COLAB_FULL_B7', 'base_logprob_preds.jsonl'))
summary = {'binding': done_all, 'common_readings': {}, 'arms': [], 'exposure_curve': {}}
for exp in EXPOSURES:
    fam = []
    for s in (0, 1, 2):
        tag = f'025_{exp}_COLAB_FULL_B7_sft_only_seed{s}'
        if not all(os.path.exists(os.path.join(D, 'results', tag, f)) for f in NEEDED):
            continue
        new = preds(tag, 'sft')
        ref = preds(f'014_p8_COLAB_FULL_B7_sft_only_seed{s}', 'sft')
        m = metric(new, ref)
        lp = _read_preds(os.path.join(D, 'results', tag, 'sft_logprob_preds.jsonl'))
        pa, pb = align_pred_rows(lp, base_probe)
        pd_, plo, phi = paired_delta_ci([r['correct'] for r in pb], [r['correct'] for r in pa])
        probe_class = ('unchanged (equivalence rule met)' if plo > -0.02 and phi < 0.02
                       else 'equivalence not established')
        # the registered erosion question is about the hard tier, so it gets the
        # same paired interval rule instead of a bare delta
        hp_new = [r['correct'] for r in pa if tier_of(r['subject']) == 'hard']
        hp_ref = [r['correct'] for r in pb if tier_of(r['subject']) == 'hard']
        hpd, hplo, hphi = paired_delta_ci(hp_ref, hp_new)
        hard_probe_class = ('unchanged (equivalence rule met)' if hplo > -0.02 and hphi < 0.02
                            else 'equivalence not established')
        fam.append({'seed': s, 'vs_p8': m, 'acc': acc(new), 'hard': hard_acc(new),
                    'probe_acc': acc(lp), 'hard_probe': hard_acc(lp),
                    'probe_delta_vs_base': pd_, 'probe_ci': [plo, phi], 'probe_class': probe_class,
                    'hard_probe_delta_vs_base': hpd, 'hard_probe_ci': [hplo, hphi],
                    'hard_probe_class': hard_probe_class,
                    'hard_margin': hard_acc(new) - hard_acc(lp)})
    if done_all and fam:
        adj = holm([a['vs_p8']['p_raw'] for a in fam])
        cats = []
        for a, p_adj in zip(fam, adj):
            a['vs_p8']['holm_p'] = p_adj
            a['vs_p8']['category'] = classify(a['vs_p8'], p_adj)
            cats.append(a['vs_p8']['category'])
        summary['common_readings'][exp] = common_reading(cats)
    else:
        for a in fam:
            a['vs_p8']['holm_p'] = None
            a['vs_p8']['category'] = 'PROVISIONAL'
    for a in fam:
        print(f"{exp} seed {a['seed']}: vs p8 {a['vs_p8']['delta']*100:+.1f}pt "
              f"(ci [{a['vs_p8']['ci'][0]*100:+.1f},{a['vs_p8']['ci'][1]*100:+.1f}]) -> {a['vs_p8']['category']}")
        print(f"   acc {a['acc']:.4f}  hard {a['hard']:.3f}  probe {a['probe_acc']:.4f} "
              f"({a['probe_delta_vs_base']*100:+.1f} vs base, {a['probe_class']}) "
              f"hard-probe {a['hard_probe']:.3f} ({a['hard_probe_delta_vs_base']*100:+.1f} "
              f"ci [{a['hard_probe_ci'][0]*100:+.1f},{a['hard_probe_ci'][1]*100:+.1f}], {a['hard_probe_class']})  "
              f"margin {a['hard_margin']*100:+.1f}")
    summary['arms'].extend({'exposure': exp, **a} for a in fam)

# the descriptive four-point curve joins the recorded p8 and p16 values
for label, fam_tag, cond in (('p8', '014_p8_COLAB_FULL_B7_sft_only_seed{s}', 'sft'),
                             ('p16', '024_p16_COLAB_FULL_B7_sft_only_seed{s}', 'sft')):
    vals = []
    for s in (0, 1, 2):
        rows = preds(fam_tag.format(s=s), cond)
        lp = _read_preds(os.path.join(D, 'results', fam_tag.format(s=s), 'sft_logprob_preds.jsonl'))
        vals.append({'seed': s, 'acc': acc(rows), 'hard': hard_acc(rows),
                     'probe_acc': acc(lp), 'hard_probe': hard_acc(lp),
                     'hard_margin': hard_acc(rows) - hard_acc(lp)})
    summary['exposure_curve'][label] = vals
for exp in EXPOSURES:
    summary['exposure_curve'][exp] = [
        {k: a[k] for k in ('seed', 'acc', 'hard', 'probe_acc', 'hard_probe', 'hard_margin')}
        for a in summary['arms'] if a['exposure'] == exp]
if done_all:
    print('\nexposure curve (mean of 3 seeds):')
    for label in ('p2', 'p8', 'p16', 'p32'):
        vs = summary['exposure_curve'][label]
        print(f"  {label:>3}: acc {sum(v['acc'] for v in vs)/3:.4f}  hard {sum(v['hard'] for v in vs)/3:.3f}  "
              f"probe {sum(v['probe_acc'] for v in vs)/3:.4f}  hard-probe {sum(v['hard_probe'] for v in vs)/3:.3f}  "
              f"margin {sum(v['hard_margin'] for v in vs)/3*100:+.1f}pt")

with open(os.path.join(D, 'results', '025_comparison_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('comparison saved, binding =', done_all)

p2 seed 0: vs p8 -1.2pt (ci [-2.3,-0.1]) -> inconclusive
   acc 0.7739  hard 0.548  probe 0.7994 (-0.1 vs base, unchanged (equivalence rule met)) hard-probe 0.571 (+0.5 ci [-1.3,+2.3], equivalence not established)  margin -2.3
p2 seed 1: vs p8 -0.7pt (ci [-1.8,+0.3]) -> equivalent
   acc 0.7728  hard 0.523  probe 0.8010 (+0.1 vs base, unchanged (equivalence rule met)) hard-probe 0.573 (+0.8 ci [-1.3,+2.8], equivalence not established)  margin -5.1
p2 seed 2: vs p8 -0.7pt (ci [-1.8,+0.4]) -> equivalent
   acc 0.7776  hard 0.538  probe 0.7999 (-0.1 vs base, unchanged (equivalence rule met)) hard-probe 0.571 (+0.5 ci [-1.3,+2.3], equivalence not established)  margin -3.3
p32 seed 0: vs p8 -0.1pt (ci [-0.9,+0.7]) -> equivalent
   acc 0.7856  hard 0.553  probe 0.7983 (-0.2 vs base, unchanged (equivalence rule met)) hard-probe 0.566 (+0.0 ci [-2.8,+2.8], equivalence not established)  margin -1.3
p32 seed 1: vs p8 +0.5pt (ci [-0.3,+1.4]) -> equivalent
   acc 0.7856  hard 0.543  probe 0.7972 (

## 14. Close the units record

In [ ]:
# close the compute-unit record for this session
import json, os, time
cu_after = 42.16    # fill from the resources panel
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run section 0 first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            record = json.load(f)
        record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        record['colab_cu_after'] = cu_after
        record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
        record['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(record) + '\n')
        os.remove(open_path)
        print('logged:', record)


reminder: fill cu_after for the record
